In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

taxiDB = pd.read_csv('taxi_dataset.csv')
taxiDB.head()

In [ ]:
taxiDB.head(5)

### Данные и целевая переменная

В таблице есть идентификатор поездки (`id`), перевозчик (`vendor_id`), время начала и окончания (`pickup_datetime`, `dropoff_datetime`), число пассажиров (`passenger_count`), координаты обеих точек и флаг сохранения записи (`store_and_fwd_flag`).

Получите `trip_duration`: переведите даты в datetime и вычислите длительность поездки в секундах. Для этого подходят `pd.to_datetime` и `.dt.total_seconds()`.

In [ ]:
### Your code is here
taxiDB['pickup_datetime'] = pd.to_datetime(taxiDB['pickup_datetime'])
taxiDB['dropoff_datetime'] = pd.to_datetime(taxiDB['dropoff_datetime'])

# 2. Теперь считаем разницу в секундах
taxiDB['trip_duration'] = (taxiDB['dropoff_datetime'] - taxiDB['pickup_datetime']).dt.total_seconds()
taxiDB.head()


### Исключение времени окончания

Уберите `dropoff_datetime`: при прогнозировании длительности новой поездки фактическое время её окончания ещё неизвестно.

In [ ]:
### Your code is here
taxiDB = taxiDB.drop(columns=['dropoff_datetime'])
taxiDB.head()

### Бинарный признак перевозчика

Переведите значения `vendor_id` из {1, 2} в {0, 1}. Далее будут подготовлены остальные признаки для модели.

In [ ]:
taxiDB['vendor_id'] = taxiDB['vendor_id'] - 1
taxiDB.head()

Найдите второй бинарный признак и представьте его значениями 0 и 1.

In [ ]:
### Your code is here
taxiDB['store_and_fwd_flag'] = taxiDB['store_and_fwd_flag'].map({'N': 0, 'Y': 1})
taxiDB.head()

In [ ]:
# Промежуточный результат: первые 10 строк, CSV с разделителем ';' (задание 6).
taxiDB.head(10).to_csv('assignment_6.csv', sep=';', index=True)

### Расстояние между точками

Разности координат можно использовать для приближённой оценки расстояния. Сначала приведите широту и долготу к километрам с учётом разных масштабов градуса, затем примените теорему Пифагора:

$$R=\sqrt{\delta_{long}^2+\delta_{lat}^2}.$$

Это приближение расстояния между точками, а не длина маршрута автомобиля.

Соберите значения широты для всех начальных и конечных точек.

In [ ]:
allLat  = list(taxiDB['pickup_latitude']) + list(taxiDB['dropoff_latitude'])

Найдите медиану объединённого списка широт — центральное значение отсортированных наблюдений.

In [ ]:
medianLat  = sorted(allLat)[int(len(allLat)/2)]

Вычтите медиану из широт и переведите полученные отклонения в километры.

In [ ]:
latMultiplier  = 111.32

taxiDB['pickup_latitude']   = latMultiplier  * (taxiDB['pickup_latitude']   - medianLat)
taxiDB['dropoff_latitude']   = latMultiplier  * (taxiDB['dropoff_latitude']  - medianLat)

In [ ]:
taxiDB.head()

После преобразования широты знак и величина значения описывают смещение к северу или югу относительно выбранной медианной широты.

In [ ]:
allLong = list(taxiDB['pickup_longitude']) + list(taxiDB['dropoff_longitude'])

medianLong  = sorted(allLong)[int(len(allLong)/2)]

longMultiplier = np.cos(medianLat*(np.pi/180.0)) * 111.32

Аналогично преобразуйте долготы, используя рассчитанные медиану и множитель перевода.

In [ ]:
taxiDB['pickup_longitude'] = longMultiplier * (taxiDB['pickup_longitude'] - medianLong)
taxiDB['dropoff_longitude'] = longMultiplier * (taxiDB['dropoff_longitude'] - medianLong)
taxiDB.head()
### Your code is here

### Зачем центрировать координаты

Отсчёт от медианной точки позволяет трактовать координаты как смещения начала и конца поездки относительно центра наблюдений. Из этих смещений можно строить дополнительные признаки; в этом задании они дальше не используются.

Рассчитайте `distance_km` как евклидово расстояние между преобразованными координатами начала и конца поездки.

In [ ]:
### Your code is here
taxiDB['distance_km'] = (
    (taxiDB['dropoff_longitude'] - taxiDB['pickup_longitude']) ** 2 +
    (taxiDB['dropoff_latitude'] - taxiDB['pickup_latitude']) ** 2
) ** 0.5

taxiDB.head()

In [ ]:
taxiDB.head()

Удалите четыре исходные колонки координат после расчёта расстояния.

In [ ]:
taxiDB = taxiDB.drop(['pickup_longitude', 'dropoff_longitude',
                      'pickup_latitude', 'dropoff_latitude'], axis=1)

In [ ]:
taxiDB.head()

In [ ]:
# Промежуточный результат: первые 10 строк после расчёта расстояния, CSV ';' (задание 7).
taxiDB.head(10).to_csv('assignment_7.csv', sep=';', index=True)

### Число пассажиров

Посмотрите, какие значения встречаются в `passenger_count`.

In [ ]:
### Your code is here
taxiDB['passenger_count'].unique()

Рассмотрите `passenger_count` как категориальный признак. Для каждой категории вычислите среднюю длительность поездки и замените исходный столбец колонкой `category_encoded` — это mean target encoding.

Такой выбор способа представления признака нужно сравнивать экспериментально с другими вариантами. В упражнении отрабатывается именно кодирование средним таргетом.

In [ ]:
### Your code is here
# 1. Считаем среднее значение trip_duration для каждого passenger_count
means = taxiDB.groupby('passenger_count')['trip_duration'].mean()

# 2. Создаем новую колонку category_encoded, подставляя вычисленные средние значения
taxiDB['category_encoded'] = taxiDB['passenger_count'].map(means)

# 3. Удаляем исходную колонку passenger_count, как требуется в задании
taxiDB = taxiDB.drop(columns=['passenger_count'])
taxiDB.head()

In [ ]:
# Промежуточный результат после кодирования: первые 10 строк, CSV ';' (задание 8).
taxiDB.head(10).to_csv('assignment_8.csv', sep=';', index=True)

### Индекс таблицы

Сделайте `id` индексом полученного DataFrame. Колонка `pickup_datetime` пока остаётся в данных.

In [ ]:
taxiDB = taxiDB.set_index('id')

In [ ]:
taxiDB.head()

In [ ]:
# Итоговый фрагмент для задания 9: первые 10 000 строк, CSV ';'.
taxiDB.head(10000).to_csv('assignment_9.csv', sep=';', index=True)